In [ ]:
import cv2 as cv
import numpy as np
from ultralytics import YOLO

# YOLOv8 모델 로드
model = YOLO('C:/Users/AI-LHJ/Desktop/ultralytics-main/runs/segment/train5/weights/best.pt')

# 이미지 읽기
img1 = cv.imread('C:/Users/AI-LHJ/Downloads/source/ch5/wire12.jpg')
img2 = cv.imread('C:/Users/AI-LHJ/Downloads/source/ch5/wire123.jpg')

# YOLOv8로 wire 객체 검출
results1 = model(img1)
results2 = model(img2)

def get_wire_boxes(results):
    boxes = []
    for r in results:
        for box in r.boxes:
            if int(box.cls) == 0:  # wire 클래스가 0번째 인덱스라고 가정
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                boxes.append([x1, y1, x2 - x1, y2 - y1])
    return boxes

boxes1 = get_wire_boxes(results1)
boxes2 = get_wire_boxes(results2)

def calculate_iou(box1, box2):
    x1, y1, w1, h1 = box1
    x2, y2, w2, h2 = box2
    
    xi1 = max(x1, x2)
    yi1 = max(y1, y2)
    xi2 = min(x1 + w1, x2 + w2)
    yi2 = min(y1 + h1, y2 + h2)
    
    inter_area = max(0, xi2 - xi1) * max(0, yi2 - yi1)
    
    box1_area = w1 * h1
    box2_area = w2 * h2
    
    iou = inter_area / float(box1_area + box2_area - inter_area)
    return iou

def filter_wire_boxes(boxes, min_aspect_ratio=3):
    filtered_boxes = []
    for box in boxes:
        x, y, w, h = box
        aspect_ratio = max(w, h) / min(w, h)
        if aspect_ratio > min_aspect_ratio:
            filtered_boxes.append(box)
    return filtered_boxes

def analyze_changes(boxes1, boxes2, iou_threshold=0.3, coord_diff_threshold=20):
    changes = []
    
    filtered_boxes1 = filter_wire_boxes(boxes1)
    filtered_boxes2 = filter_wire_boxes(boxes2)
    
    if len(filtered_boxes1) != len(filtered_boxes2):
        changes.append(('count_change', (len(filtered_boxes1), len(filtered_boxes2))))
    
    matched_boxes2 = set()
    
    for i, box1 in enumerate(filtered_boxes1):
        matched = False
        for j, box2 in enumerate(filtered_boxes2):
            iou = calculate_iou(box1, box2)
            coord_diff = np.sum(np.abs(np.array(box1) - np.array(box2)))
            if iou > iou_threshold and coord_diff < coord_diff_threshold:
                matched = True
                matched_boxes2.add(j)
                break
        if not matched:
            changes.append(('removed', box1))
    
    for j, box2 in enumerate(filtered_boxes2):
        if j not in matched_boxes2:
            changes.append(('added', box2))
    
    return changes

changes = analyze_changes(boxes1, boxes2)

# 결과 시각화
result_img = img2.copy()

# 모든 와이어 박스 표시
for box in filter_wire_boxes(boxes2):
    x, y, w, h = box
    cv.rectangle(result_img, (x, y), (x+w, y+h), (255, 255, 0), 2)  # 노란색으로 모든 와이어 표시

# 변화 표시
for change_type, change_info in changes:
    if change_type == 'count_change':
        print(f"Wire count changed: Image 1 has {change_info[0]} wires, Image 2 has {change_info[1]} wires")
    elif change_type == 'added':
        x, y, w, h = change_info
        cv.rectangle(result_img, (x, y), (x+w, y+h), (0, 255, 0), 3)  # 초록색으로 새로 추가된 와이어 강조
        cv.putText(result_img, 'New', (x, y-10), cv.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
    elif change_type == 'removed':
        print(f"Removed wire: {change_info}")

# 결과 출력
print(f"변화 수: {len(changes)}")
for change_type, change_info in changes:
    print(f"{change_type}: {change_info}")

cv.imshow('Wire Changes', result_img)
cv.waitKey(0)
cv.destroyAllWindows()

In [ ]:
# YOLOv8 결과 시각화
def visualize_detection(img, results):
    for r in results:
        for box in r.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            cls = int(box.cls)
            score = box.conf[0]  # 신뢰도 점수
            label = f"Class {cls} ({score:.2f})"
            color = (0, 255, 0)  # 초록색
            cv.rectangle(img, (x1, y1), (x2, y2), color, 2)
            cv.putText(img, label, (x1, y1 - 10), cv.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)

    cv.imshow('YOLO Detection', img)
    cv.waitKey(0)
    cv.destroyAllWindows()

# 각 이미지에 대한 탐지 시각화
visualize_detection(img1.copy(), results1)
visualize_detection(img2.copy(), results2)


In [ ]:
# YOLOv8 결과에서 클래스 인덱스 출력
def print_classes(results):
    for r in results:
        for box in r.boxes:
            print(f"Detected class index: {int(box.cls)} with confidence {box.conf[0]:.2f}")

print("Image 1 classes:")
print_classes(results1)

print("Image 2 classes:")
print_classes(results2)


In [ ]:
# 탐지된 바운딩 박스 출력
def print_boxes(boxes, image_name):
    print(f"Boxes in {image_name}:")
    for box in boxes:
        print(box)

print_boxes(boxes1, "Image 1")
print_boxes(boxes2, "Image 2")